# Prepare environment

In [1]:
import json
import datetime
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from phong_dataset import PhongDataset
from unet import UNet
from ddpm import DDPM
from evaluate import evaluate
from config import DATA_DIR, DIFFUSION_MODEL_RESULTS

In [2]:
DIFFUSION_MODEL_RESULTS.mkdir(parents=True, exist_ok=True)

In [3]:
FULL_DATASET_SIZE = 3000
TESTING_DATASET_SIZE = 600
VALIDATION_DATASET_SIZE = int((FULL_DATASET_SIZE - TESTING_DATASET_SIZE) * 0.15)
TRAINING_DATASET_SIZE = FULL_DATASET_SIZE - VALIDATION_DATASET_SIZE - TESTING_DATASET_SIZE

PARAMS_DIM = 10
IMAGE_SIZE = 128

BATCH_SIZE = 16

UNET_BASE_CHANNELS_NUM = 64
DIFFUSION_STEPS_NUM = 200

EPOCHS = 5
LEARNING_RATE = 1e-4

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [5]:
def get_timestamp() -> str:
    ct = datetime.datetime.now()

    return ct.strftime("%Y-%m-%d_%H-%M-%S")

In [6]:
CURRENT_TIMESTAMP = get_timestamp()

CURRENT_RUN_DIRECTORY = Path(f"{DIFFUSION_MODEL_RESULTS}/{CURRENT_TIMESTAMP}")
CURRENT_RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)

# Load dataset

In [7]:
train_ids = list(range(0, TRAINING_DATASET_SIZE))
val_ids = list(range(TRAINING_DATASET_SIZE, FULL_DATASET_SIZE - TESTING_DATASET_SIZE))
test_ids = list(range(FULL_DATASET_SIZE - TESTING_DATASET_SIZE, FULL_DATASET_SIZE))

print(f"Training dataset size: {len(train_ids)}")
print(f"Validation dataset size: {len(val_ids)}")
print(f"Testing dataset size: {len(test_ids)}")

Training dataset size: 2040
Validation dataset size: 360
Testing dataset size: 600


In [8]:
print(f"Training dataset range: {min(train_ids)} - {max(train_ids)}")
print(f"Validation dataset range: {min(val_ids)} - {max(val_ids)}")
print(f"Testing dataset range: {min(test_ids)} - {max(test_ids)}")

Training dataset range: 0 - 2039
Validation dataset range: 2040 - 2399
Testing dataset range: 2400 - 2999


In [9]:
train_dataset = PhongDataset(DATA_DIR, train_ids)
val_dataset = PhongDataset(DATA_DIR, val_ids)
test_dataset = PhongDataset(DATA_DIR, test_ids)

In [10]:
print(f"Sample training data shape: {train_dataset[0][0].shape} (params), {train_dataset[0][1].shape} (image)")
print(f"Sample validation data shape: {val_dataset[0][0].shape} (params), {val_dataset[0][1].shape} (image)")
print(f"Sample testing data shape: {test_dataset[0][0].shape} (params), {test_dataset[0][1].shape} (image)")

Sample training data shape: torch.Size([10]) (params), torch.Size([3, 128, 128]) (image)
Sample validation data shape: torch.Size([10]) (params), torch.Size([3, 128, 128]) (image)
Sample testing data shape: torch.Size([10]) (params), torch.Size([3, 128, 128]) (image)


In [11]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [12]:
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of testing batches: {len(test_loader)}")

Number of training batches: 128
Number of validation batches: 23
Number of testing batches: 38


# Load model

In [13]:
model = UNet(param_dim=PARAMS_DIM, base_ch=UNET_BASE_CHANNELS_NUM).to(device)
ddpm = DDPM(T=DIFFUSION_STEPS_NUM, device=device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params}")

Model parameters: 4170307


# Training

In [14]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)

In [15]:
history: dict[str, list[float]] = {
    "train_loss": [],
    "val_loss": [],
    "train_lpips": [],
    "val_lpips": [],
    "train_ssim": [],
    "val_ssim": [],
    "train_hausdorff": [],
    "val_hausdorff": [],
    "lr": [],
}

best_train_loss = float("inf")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for params, images in tqdm(train_loader, desc=f"Epoch {epoch+1} / {EPOCHS}", leave=False):
        params, images = params.to(device), images.to(device)

        t = torch.randint(0, ddpm.T, (images.size(0),), device=device)

        noisy, noise = ddpm.q_sample(images, t)

        pred = model(noisy, t, params)
        loss = F.mse_loss(pred, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["lr"].append(scheduler.get_last_lr()[0])

    print(f"Epoch {epoch+1} / {EPOCHS} | Train Loss: {train_loss:.4f} | lr: {scheduler.get_last_lr()[0]:.6f}")

    if train_loss < best_train_loss:
        best_train_loss = train_loss
        torch.save(model.state_dict(), CURRENT_RUN_DIRECTORY / "best_model.pth")
        print("Best model saved.")

print(f"Training finished with best loss: {best_train_loss:.4f}")

Epoch 1 / 5 | Train Loss: 0.1836 | lr: 0.000090
Best model saved.


Epoch 2 / 5 | Train Loss: 0.0378 | lr: 0.000065
Best model saved.


Epoch 3 / 5 | Train Loss: 0.0242 | lr: 0.000035
Best model saved.


Epoch 4 / 5 | Train Loss: 0.0186 | lr: 0.000010
Best model saved.


Epoch 5 / 5 | Train Loss: 0.0167 | lr: 0.000000
Best model saved.
Training finished with best loss: 0.0167


In [16]:
history_path = Path(f"{CURRENT_RUN_DIRECTORY}/history.json")

with open(history_path, "w") as f:
    json.dump(history, f, indent=4, sort_keys=True)

# Evaluation

In [17]:
model = UNet(param_dim=PARAMS_DIM, base_ch=UNET_BASE_CHANNELS_NUM).to(device)
model.load_state_dict(torch.load(CURRENT_RUN_DIRECTORY / "best_model.pth"))
model.eval()

UNet(
  (time_emb): Sequential(
    (0): SinusoidalPosEmb()
    (1): Linear(in_features=128, out_features=256, bias=True)
    (2): SiLU()
    (3): Linear(in_features=256, out_features=128, bias=True)
  )
  (cond_emb): Sequential(
    (0): Linear(in_features=10, out_features=128, bias=True)
    (1): SiLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  )
  (enc1): ResBlock(
    (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (norm2): GroupNorm(8, 64, eps=1e-05, affine=True)
    (time_proj): Linear(in_features=128, out_features=64, bias=True)
    (cond_proj): Linear(in_features=128, out_features=64, bias=True)
    (skip): Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
  )
  (enc2): ResBlock(
    (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(128, 128, kernel_si

In [18]:
metrics = evaluate(model, ddpm, test_loader, device, CURRENT_RUN_DIRECTORY, num_samples=TESTING_DATASET_SIZE)

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/home/dominika/Desktop/26L_sem8_mgr1/SIGK/AIComputerGraphics/project3-rendering/venv/lib/python3.14/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/dominika/Desktop/26L_sem8_mgr1/SIGK/AIComputerGraphics/project3-rendering/venv/lib/python3.14/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/dominika/Desktop/26L_sem8_mgr1/SIGK/AIComputerGraphics/project3-rendering/venv/lib/python3.14/site-packages/lpips/weights/v0.1/alex.pth


Evaluating:  97%|█████████▋| 37/38 [07:12<00:11, 11.69s/it]


In [19]:
for metric_name, metric_value in metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

SSIM: 0.0006
LPIPS: 0.9312
Hausdorff: 129.3531
